<a href="https://colab.research.google.com/github/hyunkyung31/coronary-ai-ml-dl/blob/main/hyunkyung/03_AngioCAD_%ED%99%98%EC%9E%90%EB%8B%A8%EC%9C%84_%EB%8D%B0%EC%9D%B4%ED%84%B0%EB%B6%84%ED%95%A0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

분할의 핵심 원칙
- 이미지가 아니라 환자 단위로 분할
- 412명 전체를 먼저 분할
- 36명의 Clinical 미보유 환자는 image-only 실험에 유지
- Multimodal은 같은 split에서 Clinical 보유 환자만 사용
- imputation·scaling·feature selection은 분할 후 train 안에서만 수행
- 우선 seed 42 후보를 만들고 분포를 검증한 뒤 확정

In [1]:
#@title 셀 1. 마운트 및 경로 복구

#@title 03 환자 단위 데이터 분할 - 환경 설정

from google.colab import drive
from pathlib import Path

import pandas as pd
import numpy as np

drive.mount("/content/drive")

PROJECT_DIR = Path(
    "/content/drive/MyDrive/"
    "[MacGyver]최종프로젝트/"
    "03_data/AngioCAD"
)

AUDIT_DIR = PROJECT_DIR / "audit_results"

PREPROCESSING_DIR = (
    PROJECT_DIR / "preprocessing_results"
)

SPLIT_DIR = PROJECT_DIR / "split_results"

SPLIT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

LABELS_PATH = (
    PROJECT_DIR / "AngioCAD_Labels.xlsx"
)

print("Project:", PROJECT_DIR)
print("Labels:", LABELS_PATH.exists())
print("Split results:", SPLIT_DIR)

Mounted at /content/drive
Project: /content/drive/MyDrive/[MacGyver]최종프로젝트/03_data/AngioCAD
Labels: True
Split results: /content/drive/MyDrive/[MacGyver]최종프로젝트/03_data/AngioCAD/split_results


In [2]:
#@title Series 및 Clinical 결과 불러오기

paper_series_raw_df = pd.read_csv(
    AUDIT_DIR / "14_paper_model_series.csv"
)

extended_series_raw_df = pd.read_csv(
    AUDIT_DIR / "15_extended_model_series.csv"
)

series_view_raw_df = pd.read_csv(
    AUDIT_DIR / "13_series_view_manifest.csv"
)

clinical_clean_df = pd.read_csv(
    PREPROCESSING_DIR
    / "01_clinical_deterministic_clean.csv"
)

labels_raw_df = pd.read_excel(
    LABELS_PATH
)

dataframes = [
    paper_series_raw_df,
    extended_series_raw_df,
    series_view_raw_df,
    clinical_clean_df,
    labels_raw_df,
]

for dataframe in dataframes:
    dataframe.columns = [
        column.strip()
        if isinstance(column, str)
        else column
        for column in dataframe.columns
    ]

print("Paper series:", len(paper_series_raw_df))
print("Extended series:", len(extended_series_raw_df))
print("Clinical patients:", len(clinical_clean_df))
print("Labels patients:", len(labels_raw_df))

Paper series: 2676
Extended series: 2680
Clinical patients: 377
Labels patients: 413


/usr/local/lib/python3.13/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Conditional Formatting extension is not supported and will be removed
  warn(msg)


In [3]:
#@title ID와 view 표준화

SERIES_KEY_COLUMNS = [
    "patient_id",
    "series_id",
]

for dataframe in [
    paper_series_raw_df,
    extended_series_raw_df,
    series_view_raw_df,
]:
    for column in SERIES_KEY_COLUMNS:
        dataframe[column] = pd.to_numeric(
            dataframe[column],
            errors="raise",
        ).astype(int)

VIEW_COLUMN_CANDIDATES = [
    "view_status",
    "view",
    "coronary_view",
    "assigned_view",
]

view_column = next(
    (
        column
        for column in VIEW_COLUMN_CANDIDATES
        if column in series_view_raw_df.columns
    ),
    None,
)

if view_column is None:
    raise KeyError(
        "view 열을 찾지 못했습니다: "
        f"{series_view_raw_df.columns.tolist()}"
    )

series_view_lookup_df = (
    series_view_raw_df[
        SERIES_KEY_COLUMNS + [view_column]
    ]
    .rename(columns={view_column: "view"})
    .drop_duplicates(SERIES_KEY_COLUMNS)
)

series_view_lookup_df["view"] = (
    series_view_lookup_df["view"]
    .astype("string")
    .str.strip()
    .str.upper()
)

paper_series_df = (
    paper_series_raw_df[
        SERIES_KEY_COLUMNS
    ]
    .drop_duplicates()
    .merge(
        series_view_lookup_df,
        on=SERIES_KEY_COLUMNS,
        how="left",
        validate="one_to_one",
    )
)

extended_series_df = (
    extended_series_raw_df[
        SERIES_KEY_COLUMNS
    ]
    .drop_duplicates()
    .merge(
        series_view_lookup_df,
        on=SERIES_KEY_COLUMNS,
        how="left",
        validate="one_to_one",
    )
)

assert len(paper_series_df) == 2676
assert len(extended_series_df) == 2680

assert paper_series_df["view"].isin(
    ["LEFT", "RIGHT"]
).all()

assert extended_series_df["view"].isin(
    ["LEFT", "RIGHT"]
).all()

print("✅ Series ID와 view 표준화 완료")

✅ Series ID와 view 표준화 완료


In [4]:
#@title 환자별 정답 생성

STENOSIS_COLUMNS = [
    "LM",
    "Prox LAD",
    "Mid LAD",
    "Dist LAD",
    "1st dig",
    "2nd dig",
    "Prox LCX",
    "Mid LCX",
    "Dist LCX",
    "OM",
    "Prox RCA",
    "Mid RCA",
    "Dist RCA",
    "PDA",
    "PLB",
]

LEFT_ARTERIES = [
    "LM",
    "Prox LAD",
    "Mid LAD",
    "Dist LAD",
    "1st dig",
    "2nd dig",
    "Prox LCX",
    "Mid LCX",
    "Dist LCX",
    "OM",
]

RIGHT_ARTERIES = [
    "Prox RCA",
    "Mid RCA",
    "Dist RCA",
    "PDA",
    "PLB",
]

ALLOWED_VALUES = {
    "NL",
    "1-25",
    "26-50",
    "51-75",
    "76-90",
    "91-99",
    "100",
}

DEFINITE_OVER_50_VALUES = {
    "51-75",
    "76-90",
    "91-99",
    "100",
}


def normalize_stenosis_value(value):
    if pd.isna(value):
        return pd.NA

    if isinstance(value, (int, np.integer)):
        return str(int(value))

    if isinstance(value, (float, np.floating)):
        if float(value).is_integer():
            return str(int(value))

    value = str(value).strip()

    value = (
        value.replace("–", "-")
        .replace("—", "-")
        .replace("−", "-")
    )

    if value.upper() == "NL":
        return "NL"

    if value == "100.0":
        return "100"

    return value


LABEL_ID_CANDIDATES = [
    "ID",
    "patient_id",
    "Patient ID",
]

label_id_column = next(
    (
        column
        for column in LABEL_ID_CANDIDATES
        if column in labels_raw_df.columns
    ),
    None,
)

if label_id_column is None:
    raise KeyError(
        "Labels 환자 ID 열을 찾지 못했습니다."
    )

labels_df = labels_raw_df.rename(
    columns={label_id_column: "patient_id"}
).copy()

labels_df["patient_id"] = pd.to_numeric(
    labels_df["patient_id"],
    errors="raise",
).astype(int)

for column in STENOSIS_COLUMNS:
    labels_df[column] = (
        labels_df[column]
        .map(normalize_stenosis_value)
        .astype("string")
    )

    invalid_values = set(
        labels_df[column].dropna().unique()
    ) - ALLOWED_VALUES

    assert not invalid_values, (
        f"{column} 비정상 라벨: {invalid_values}"
    )


def any_stenosis_from_columns(
    dataframe,
    columns,
):
    return (
        dataframe[columns]
        .ne("NL")
        .any(axis=1)
        .astype(int)
    )


def definite_over_50_from_columns(
    dataframe,
    columns,
):
    return (
        dataframe[columns]
        .isin(DEFINITE_OVER_50_VALUES)
        .any(axis=1)
        .astype(int)
    )


patient_target_df = labels_df[
    ["patient_id"]
].copy()

patient_target_df["left_any_stenosis"] = (
    any_stenosis_from_columns(
        labels_df,
        LEFT_ARTERIES,
    )
)

patient_target_df["right_any_stenosis"] = (
    any_stenosis_from_columns(
        labels_df,
        RIGHT_ARTERIES,
    )
)

patient_target_df[
    "left_definite_over_50"
] = definite_over_50_from_columns(
    labels_df,
    LEFT_ARTERIES,
)

patient_target_df[
    "right_definite_over_50"
] = definite_over_50_from_columns(
    labels_df,
    RIGHT_ARTERIES,
)

patient_target_df["any_stenosis"] = (
    (
        patient_target_df["left_any_stenosis"]
        == 1
    )
    | (
        patient_target_df["right_any_stenosis"]
        == 1
    )
).astype(int)

patient_target_df[
    "any_definite_over_50"
] = (
    (
        patient_target_df[
            "left_definite_over_50"
        ]
        == 1
    )
    | (
        patient_target_df[
            "right_definite_over_50"
        ]
        == 1
    )
).astype(int)

print("환자 target rows:", len(patient_target_df))
print("중복 patient_id:", patient_target_df["patient_id"].duplicated().sum())

assert len(patient_target_df) == 413
assert patient_target_df["patient_id"].duplicated().sum() == 0

print("✅ 환자별 정답 생성 완료")

환자 target rows: 413
중복 patient_id: 0
✅ 환자별 정답 생성 완료


In [5]:
#@title Series에 view별 정답 연결

def attach_series_labels(
    series_df,
    patient_targets,
):
    result = series_df.merge(
        patient_targets,
        on="patient_id",
        how="left",
        validate="many_to_one",
    )

    result["view_any_stenosis"] = np.where(
        result["view"].eq("LEFT"),
        result["left_any_stenosis"],
        result["right_any_stenosis"],
    ).astype(int)

    result[
        "view_definite_over_50"
    ] = np.where(
        result["view"].eq("LEFT"),
        result["left_definite_over_50"],
        result["right_definite_over_50"],
    ).astype(int)

    return result


paper_labeled_series_df = attach_series_labels(
    paper_series_df,
    patient_target_df,
)

extended_labeled_series_df = attach_series_labels(
    extended_series_df,
    patient_target_df,
)

print(
    "Paper positive rate:",
    f"{paper_labeled_series_df['view_any_stenosis'].mean():.1%}",
)

print(
    "Extended positive rate:",
    f"{extended_labeled_series_df['view_any_stenosis'].mean():.1%}",
)

assert len(paper_labeled_series_df) == 2676
assert len(extended_labeled_series_df) == 2680

print("✅ Series별 view target 연결 완료")

Paper positive rate: 80.5%
Extended positive rate: 80.5%
✅ Series별 view target 연결 완료


In [6]:
#@title 환자 split master 생성

CLINICAL_ID_CANDIDATES = [
    "ID",
    "patient_id",
    "Patient ID",
]

clinical_id_column = next(
    (
        column
        for column in CLINICAL_ID_CANDIDATES
        if column in clinical_clean_df.columns
    ),
    None,
)

if clinical_id_column is None:
    raise KeyError(
        "Clinical 환자 ID 열을 찾지 못했습니다."
    )

clinical_patient_ids = set(
    pd.to_numeric(
        clinical_clean_df[clinical_id_column],
        errors="raise",
    ).astype(int)
)

paper_patient_ids = sorted(
    paper_labeled_series_df[
        "patient_id"
    ].unique()
)

patient_split_master_df = (
    pd.DataFrame({
        "patient_id": paper_patient_ids,
    })
    .merge(
        patient_target_df,
        on="patient_id",
        how="left",
        validate="one_to_one",
    )
)

patient_split_master_df[
    "clinical_available"
] = (
    patient_split_master_df["patient_id"]
    .isin(clinical_patient_ids)
    .astype(int)
)

series_counts_df = (
    paper_labeled_series_df.groupby(
        ["patient_id", "view"]
    )
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

for column in ["LEFT", "RIGHT"]:
    if column not in series_counts_df.columns:
        series_counts_df[column] = 0

series_counts_df = series_counts_df.rename(
    columns={
        "LEFT": "left_series_count",
        "RIGHT": "right_series_count",
    }
)

series_counts_df["total_series_count"] = (
    series_counts_df["left_series_count"]
    + series_counts_df["right_series_count"]
)

patient_split_master_df = (
    patient_split_master_df.merge(
        series_counts_df,
        on="patient_id",
        how="left",
        validate="one_to_one",
    )
)

# 좌우 stenosis 조합을 stratification 기준으로 사용
patient_split_master_df[
    "stratification_group"
] = (
    "L"
    + patient_split_master_df[
        "left_any_stenosis"
    ].astype(str)
    + "_R"
    + patient_split_master_df[
        "right_any_stenosis"
    ].astype(str)
)

print("분할 대상 환자:", len(patient_split_master_df))
print(
    "Clinical 보유:",
    patient_split_master_df[
        "clinical_available"
    ].sum(),
)

display(
    patient_split_master_df[
        "stratification_group"
    ]
    .value_counts()
    .rename("patient_count")
    .to_frame()
)

assert len(patient_split_master_df) == 412
assert (
    patient_split_master_df[
        "clinical_available"
    ].sum()
    == 376
)

print("✅ 환자 split master 생성 완료")

분할 대상 환자: 412
Clinical 보유: 376


,patient_count
stratification_group,
L1_R1,276
L1_R0,68
L0_R0,53
L0_R1,15


✅ 환자 split master 생성 완료


In [7]:
#@title Seed 42 분할 후보 생성

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
)

RANDOM_STATE = 42

all_indices = np.arange(
    len(patient_split_master_df)
)

development_indices, test_indices = (
    train_test_split(
        all_indices,
        test_size=0.20,
        random_state=RANDOM_STATE,
        shuffle=True,
        stratify=patient_split_master_df[
            "stratification_group"
        ],
    )
)

patient_split_candidate_df = (
    patient_split_master_df.copy()
)

patient_split_candidate_df[
    "final_split"
] = "DEVELOPMENT"

patient_split_candidate_df.loc[
    test_indices,
    "final_split",
] = "TEST"

patient_split_candidate_df[
    "development_cv_fold"
] = pd.Series(
    pd.NA,
    index=patient_split_candidate_df.index,
    dtype="Int64",
)

development_df = (
    patient_split_candidate_df.loc[
        development_indices
    ]
    .copy()
)

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

for fold, (_, validation_position) in enumerate(
    skf.split(
        development_df,
        development_df[
            "stratification_group"
        ],
    )
):
    validation_indices = (
        development_df.iloc[
            validation_position
        ].index
    )

    patient_split_candidate_df.loc[
        validation_indices,
        "development_cv_fold",
    ] = fold

print(
    patient_split_candidate_df[
        "final_split"
    ].value_counts()
)

print("\nDevelopment CV fold 크기")

display(
    patient_split_candidate_df.loc[
        patient_split_candidate_df[
            "final_split"
        ].eq("DEVELOPMENT")
    ]
    ["development_cv_fold"]
    .value_counts()
    .sort_index()
    .rename("patient_count")
    .to_frame()
)

assert patient_split_candidate_df.loc[
    patient_split_candidate_df[
        "final_split"
    ].eq("TEST"),
    "development_cv_fold",
].isna().all()

assert patient_split_candidate_df.loc[
    patient_split_candidate_df[
        "final_split"
    ].eq("DEVELOPMENT"),
    "development_cv_fold",
].notna().all()

print("✅ Seed 42 환자 단위 분할 후보 생성 완료")

final_split
DEVELOPMENT    329
TEST            83
Name: count, dtype: int64

Development CV fold 크기


,patient_count
development_cv_fold,
0,66
1,66
2,66
3,66
4,65


✅ Seed 42 환자 단위 분할 후보 생성 완료


In [8]:
#@title 분포 검증

patient_distribution_df = (
    patient_split_candidate_df.groupby(
        "final_split"
    )
    .agg(
        patient_count=(
            "patient_id",
            "nunique",
        ),
        left_positive_rate=(
            "left_any_stenosis",
            "mean",
        ),
        right_positive_rate=(
            "right_any_stenosis",
            "mean",
        ),
        any_positive_rate=(
            "any_stenosis",
            "mean",
        ),
        definite_over_50_rate=(
            "any_definite_over_50",
            "mean",
        ),
        clinical_available_count=(
            "clinical_available",
            "sum",
        ),
        clinical_available_rate=(
            "clinical_available",
            "mean",
        ),
        mean_series_per_patient=(
            "total_series_count",
            "mean",
        ),
    )
    .reset_index()
)

display(patient_distribution_df)

print("\n=== Stratification group 분포 ===")

display(
    pd.crosstab(
        patient_split_candidate_df[
            "final_split"
        ],
        patient_split_candidate_df[
            "stratification_group"
        ],
        margins=True,
    )
)

paper_split_candidate_df = (
    paper_labeled_series_df.merge(
        patient_split_candidate_df[
            [
                "patient_id",
                "final_split",
                "development_cv_fold",
            ]
        ],
        on="patient_id",
        how="left",
        validate="many_to_one",
    )
)

series_distribution_df = (
    paper_split_candidate_df.groupby(
        ["final_split", "view"]
    )
    .agg(
        series_count=(
            "series_id",
            "size",
        ),
        positive_count=(
            "view_any_stenosis",
            "sum",
        ),
        positive_rate=(
            "view_any_stenosis",
            "mean",
        ),
        definite_over_50_rate=(
            "view_definite_over_50",
            "mean",
        ),
    )
    .reset_index()
)

print("\n=== Series 분포 ===")
display(series_distribution_df)

CANDIDATE_PATH = (
    SPLIT_DIR
    / "01_patient_split_candidate_seed42.csv"
)

patient_split_candidate_df.to_csv(
    CANDIDATE_PATH,
    index=False,
    encoding="utf-8-sig",
)

print("후보 저장:", CANDIDATE_PATH)

,final_split,patient_count,left_positive_rate,right_positive_rate,any_positive_rate,definite_over_50_rate,clinical_available_count,clinical_available_rate,mean_series_per_patient
0,DEVELOPMENT,329,0.835866,0.708207,0.87234,0.778116,303,0.920973,6.565350
1,TEST,83,0.831325,0.698795,0.86747,0.807229,73,0.879518,6.216867



=== Stratification group 분포 ===


stratification_group,L0_R0,L0_R1,L1_R0,L1_R1,All
final_split,,,,,
DEVELOPMENT,42,12,54,221,329
TEST,11,3,14,55,83
All,53,15,68,276,412



=== Series 분포 ===


,final_split,view,series_count,positive_count,positive_rate,definite_over_50_rate
0,DEVELOPMENT,LEFT,1503,1262,0.839654,0.715236
1,DEVELOPMENT,RIGHT,657,476,0.724505,0.445967
2,TEST,LEFT,360,305,0.847222,0.747222
3,TEST,RIGHT,156,111,0.711538,0.455128


후보 저장: /content/drive/MyDrive/[MacGyver]최종프로젝트/03_data/AngioCAD/split_results/01_patient_split_candidate_seed42.csv


In [9]:
#@title Seed 42 환자 split 최종 확정

SPLIT_VERSION = "patient_split_v1_seed42"

patient_split_final_df = (
    patient_split_candidate_df
    .copy()
    .sort_values("patient_id")
    .reset_index(drop=True)
)

patient_split_final_df.insert(
    0,
    "split_version",
    SPLIT_VERSION,
)

development_patient_ids = set(
    patient_split_final_df.loc[
        patient_split_final_df[
            "final_split"
        ].eq("DEVELOPMENT"),
        "patient_id",
    ]
)

test_patient_ids = set(
    patient_split_final_df.loc[
        patient_split_final_df[
            "final_split"
        ].eq("TEST"),
        "patient_id",
    ]
)

patient_overlap = (
    development_patient_ids
    & test_patient_ids
)

print("Development 환자:", len(development_patient_ids))
print("Test 환자:", len(test_patient_ids))
print("환자 중복:", len(patient_overlap))

assert len(patient_split_final_df) == 412
assert len(development_patient_ids) == 329
assert len(test_patient_ids) == 83
assert len(patient_overlap) == 0
assert patient_split_final_df["patient_id"].duplicated().sum() == 0

print("✅ 환자 split 최종 확정")

Development 환자: 329
Test 환자: 83
환자 중복: 0
✅ 환자 split 최종 확정


In [10]:
#@title Series와 frame에 split 연결

assignment_columns = [
    "patient_id",
    "final_split",
    "development_cv_fold",
    "clinical_available",
    "stratification_group",
]

patient_assignment_df = (
    patient_split_final_df[
        assignment_columns
    ]
)

paper_series_split_df = (
    paper_labeled_series_df.merge(
        patient_assignment_df,
        on="patient_id",
        how="left",
        validate="many_to_one",
    )
)

extended_series_split_df = (
    extended_labeled_series_df.merge(
        patient_assignment_df,
        on="patient_id",
        how="left",
        validate="many_to_one",
    )
)

eligible_frame_manifest_df = pd.read_csv(
    PREPROCESSING_DIR
    / "03_eligible_frame_manifest.csv"
)

eligible_frame_manifest_df["patient_id"] = (
    pd.to_numeric(
        eligible_frame_manifest_df["patient_id"],
        errors="raise",
    ).astype(int)
)

eligible_frame_manifest_df["series_id"] = (
    pd.to_numeric(
        eligible_frame_manifest_df["series_id"],
        errors="raise",
    ).astype(int)
)

eligible_frame_split_df = (
    eligible_frame_manifest_df.merge(
        patient_assignment_df,
        on="patient_id",
        how="left",
        validate="many_to_one",
    )
)

multimodal_patient_split_df = (
    patient_split_final_df.loc[
        patient_split_final_df[
            "clinical_available"
        ].eq(1)
    ]
    .copy()
    .reset_index(drop=True)
)

print("Paper series:", len(paper_series_split_df))
print("Extended series:", len(extended_series_split_df))
print("Extended frames:", len(eligible_frame_split_df))
print(
    "Multimodal 환자:",
    len(multimodal_patient_split_df),
)

assert len(paper_series_split_df) == 2676
assert len(extended_series_split_df) == 2680
assert len(eligible_frame_split_df) == 119046
assert len(multimodal_patient_split_df) == 376

assert paper_series_split_df[
    "final_split"
].notna().all()

assert extended_series_split_df[
    "final_split"
].notna().all()

assert eligible_frame_split_df[
    "final_split"
].notna().all()

print("✅ Series/frame/multimodal split 연결 완료")

Paper series: 2676
Extended series: 2680
Extended frames: 119046
Multimodal 환자: 376
✅ Series/frame/multimodal split 연결 완료


In [11]:
#@title Development 5-fold 분포 검증

development_patient_df = (
    patient_split_final_df.loc[
        patient_split_final_df[
            "final_split"
        ].eq("DEVELOPMENT")
    ]
    .copy()
)

development_patient_fold_df = (
    development_patient_df.groupby(
        "development_cv_fold",
        dropna=False,
    )
    .agg(
        patient_count=(
            "patient_id",
            "nunique",
        ),
        left_positive_rate=(
            "left_any_stenosis",
            "mean",
        ),
        right_positive_rate=(
            "right_any_stenosis",
            "mean",
        ),
        any_positive_rate=(
            "any_stenosis",
            "mean",
        ),
        definite_over_50_rate=(
            "any_definite_over_50",
            "mean",
        ),
        clinical_available_count=(
            "clinical_available",
            "sum",
        ),
        clinical_available_rate=(
            "clinical_available",
            "mean",
        ),
        mean_series_per_patient=(
            "total_series_count",
            "mean",
        ),
    )
    .reset_index()
)

display(development_patient_fold_df)

print("\n=== Development fold별 환자 strata ===")

development_strata_table = pd.crosstab(
    development_patient_df[
        "development_cv_fold"
    ],
    development_patient_df[
        "stratification_group"
    ],
)

display(development_strata_table)

development_series_df = (
    paper_series_split_df.loc[
        paper_series_split_df[
            "final_split"
        ].eq("DEVELOPMENT")
    ]
    .copy()
)

development_series_fold_df = (
    development_series_df.groupby(
        ["development_cv_fold", "view"]
    )
    .agg(
        patient_count=(
            "patient_id",
            "nunique",
        ),
        series_count=(
            "series_id",
            "size",
        ),
        positive_count=(
            "view_any_stenosis",
            "sum",
        ),
        positive_rate=(
            "view_any_stenosis",
            "mean",
        ),
        definite_over_50_rate=(
            "view_definite_over_50",
            "mean",
        ),
    )
    .reset_index()
)

print("\n=== Development fold별 series 분포 ===")
display(development_series_fold_df)

# 모든 fold에 네 strata가 존재하는지 확인
assert development_strata_table.shape == (5, 4)
assert (development_strata_table > 0).all().all()

print("✅ Development 5-fold strata 검증 통과")

,development_cv_fold,patient_count,left_positive_rate,right_positive_rate,any_positive_rate,definite_over_50_rate,clinical_available_count,clinical_available_rate,mean_series_per_patient
0,0,66,0.848485,0.712121,0.878788,0.772727,58,0.878788,6.257576
1,1,66,0.833333,0.696970,0.863636,0.727273,60,0.909091,6.833333
2,2,66,0.818182,0.712121,0.863636,0.727273,61,0.924242,6.530303
3,3,66,0.833333,0.712121,0.878788,0.863636,62,0.939394,6.681818
4,4,65,0.846154,0.707692,0.876923,0.800000,62,0.953846,6.523077



=== Development fold별 환자 strata ===


stratification_group,L0_R0,L0_R1,L1_R0,L1_R1
development_cv_fold,,,,
0,8,2,11,45
1,9,2,11,44
2,9,3,10,44
3,8,3,11,44
4,8,2,11,44



=== Development fold별 series 분포 ===


,development_cv_fold,view,patient_count,series_count,positive_count,positive_rate,definite_over_50_rate
0,0,LEFT,65,279,231,0.827957,0.681004
1,0,RIGHT,66,134,97,0.723881,0.470149
2,1,LEFT,66,317,267,0.842271,0.681388
3,1,RIGHT,65,134,102,0.761194,0.477612
4,2,LEFT,66,292,239,0.818493,0.671233
5,2,RIGHT,66,139,92,0.661871,0.453237
6,3,LEFT,65,316,270,0.854430,0.806962
7,3,RIGHT,64,125,93,0.744000,0.448000
8,4,LEFT,65,299,255,0.852843,0.729097
9,4,RIGHT,65,125,92,0.736000,0.376000


✅ Development 5-fold strata 검증 통과


In [12]:
#@title Split 균형 정량 판정

development_summary = (
    patient_distribution_df.loc[
        patient_distribution_df[
            "final_split"
        ].eq("DEVELOPMENT")
    ].iloc[0]
)

test_summary = (
    patient_distribution_df.loc[
        patient_distribution_df[
            "final_split"
        ].eq("TEST")
    ].iloc[0]
)

balance_records = []


def add_balance_check(
    metric,
    development_value,
    test_value,
    threshold,
):
    difference = abs(
        float(development_value)
        - float(test_value)
    )

    balance_records.append({
        "metric": metric,
        "development_value": float(
            development_value
        ),
        "test_value": float(test_value),
        "absolute_difference": difference,
        "allowed_difference": threshold,
        "status": (
            "PASS"
            if difference <= threshold
            else "FAIL"
        ),
    })


add_balance_check(
    "left_positive_rate",
    development_summary["left_positive_rate"],
    test_summary["left_positive_rate"],
    0.03,
)

add_balance_check(
    "right_positive_rate",
    development_summary["right_positive_rate"],
    test_summary["right_positive_rate"],
    0.03,
)

add_balance_check(
    "any_positive_rate",
    development_summary["any_positive_rate"],
    test_summary["any_positive_rate"],
    0.03,
)

add_balance_check(
    "definite_over_50_rate",
    development_summary[
        "definite_over_50_rate"
    ],
    test_summary[
        "definite_over_50_rate"
    ],
    0.05,
)

add_balance_check(
    "clinical_available_rate",
    development_summary[
        "clinical_available_rate"
    ],
    test_summary[
        "clinical_available_rate"
    ],
    0.06,
)

for view in ["LEFT", "RIGHT"]:
    development_view = (
        series_distribution_df.loc[
            (
                series_distribution_df[
                    "final_split"
                ].eq("DEVELOPMENT")
            )
            & (
                series_distribution_df[
                    "view"
                ].eq(view)
            )
        ].iloc[0]
    )

    test_view = (
        series_distribution_df.loc[
            (
                series_distribution_df[
                    "final_split"
                ].eq("TEST")
            )
            & (
                series_distribution_df[
                    "view"
                ].eq(view)
            )
        ].iloc[0]
    )

    add_balance_check(
        f"{view.lower()}_series_positive_rate",
        development_view["positive_rate"],
        test_view["positive_rate"],
        0.05,
    )

    add_balance_check(
        f"{view.lower()}_series_definite_rate",
        development_view[
            "definite_over_50_rate"
        ],
        test_view[
            "definite_over_50_rate"
        ],
        0.05,
    )

split_balance_audit_df = pd.DataFrame(
    balance_records
)

display(split_balance_audit_df)

assert split_balance_audit_df[
    "status"
].eq("PASS").all()

print("✅ Development/Test 균형 기준 통과")

,metric,development_value,test_value,absolute_difference,allowed_difference,status
0,left_positive_rate,0.835866,0.831325,0.004541,0.03,PASS
1,right_positive_rate,0.708207,0.698795,0.009412,0.03,PASS
2,any_positive_rate,0.872340,0.867470,0.004871,0.03,PASS
3,definite_over_50_rate,0.778116,0.807229,0.029113,0.05,PASS
4,clinical_available_rate,0.920973,0.879518,0.041455,0.06,PASS
5,left_series_positive_rate,0.839654,0.847222,0.007568,0.05,PASS
6,left_series_definite_rate,0.715236,0.747222,0.031986,0.05,PASS
7,right_series_positive_rate,0.724505,0.711538,0.012967,0.05,PASS
8,right_series_definite_rate,0.445967,0.455128,0.009162,0.05,PASS


✅ Development/Test 균형 기준 통과


In [13]:
#@title 누수 방지 최종 Gate

paper_duplicate_series = (
    paper_series_split_df.duplicated(
        ["patient_id", "series_id"]
    ).sum()
)

extended_duplicate_series = (
    extended_series_split_df.duplicated(
        ["patient_id", "series_id"]
    ).sum()
)

duplicate_frame_keys = (
    eligible_frame_split_df.duplicated(
        [
            "patient_id",
            "series_id",
            "frame_index",
        ]
    ).sum()
)

test_with_cv_fold = (
    patient_split_final_df.loc[
        patient_split_final_df[
            "final_split"
        ].eq("TEST"),
        "development_cv_fold",
    ]
    .notna()
    .sum()
)

development_without_fold = (
    patient_split_final_df.loc[
        patient_split_final_df[
            "final_split"
        ].eq("DEVELOPMENT"),
        "development_cv_fold",
    ]
    .isna()
    .sum()
)

split_gate_df = pd.DataFrame([
    {
        "check": "patient_overlap_dev_test",
        "observed": len(patient_overlap),
        "expected": 0,
    },
    {
        "check": "duplicate_patient_id",
        "observed": (
            patient_split_final_df[
                "patient_id"
            ].duplicated().sum()
        ),
        "expected": 0,
    },
    {
        "check": "paper_duplicate_series",
        "observed": paper_duplicate_series,
        "expected": 0,
    },
    {
        "check": "extended_duplicate_series",
        "observed": extended_duplicate_series,
        "expected": 0,
    },
    {
        "check": "duplicate_frame_key",
        "observed": duplicate_frame_keys,
        "expected": 0,
    },
    {
        "check": "test_patient_with_cv_fold",
        "observed": test_with_cv_fold,
        "expected": 0,
    },
    {
        "check": "development_patient_without_fold",
        "observed": development_without_fold,
        "expected": 0,
    },
    {
        "check": "balance_failed_check",
        "observed": (
            split_balance_audit_df[
                "status"
            ].ne("PASS").sum()
        ),
        "expected": 0,
    },
])

split_gate_df["status"] = np.where(
    split_gate_df["observed"]
    == split_gate_df["expected"],
    "PASS",
    "FAIL",
)

display(split_gate_df)

assert split_gate_df["status"].eq("PASS").all()

print("✅ 환자 단위 데이터 분할 Gate 통과")
print("✅ Development/Test 및 CV fold 누수 없음")

,check,observed,expected,status
0,patient_overlap_dev_test,0,0,PASS
1,duplicate_patient_id,0,0,PASS
2,paper_duplicate_series,0,0,PASS
3,extended_duplicate_series,0,0,PASS
4,duplicate_frame_key,0,0,PASS
5,test_patient_with_cv_fold,0,0,PASS
6,development_patient_without_fold,0,0,PASS
7,balance_failed_check,0,0,PASS


✅ 환자 단위 데이터 분할 Gate 통과
✅ Development/Test 및 CV fold 누수 없음


In [14]:
#@title Split hash와 최종 파일 저장

import hashlib
import json

canonical_split_text = (
    patient_split_final_df[
        [
            "patient_id",
            "final_split",
            "development_cv_fold",
        ]
    ]
    .sort_values("patient_id")
    .to_csv(
        index=False,
        lineterminator="\n",
    )
)

split_sha256 = hashlib.sha256(
    canonical_split_text.encode("utf-8")
).hexdigest()

split_config = {
    "split_version": SPLIT_VERSION,
    "random_state": RANDOM_STATE,
    "test_size": 0.20,
    "development_cv_folds": 5,
    "split_unit": "patient",
    "stratification": (
        "left_any_stenosis + right_any_stenosis"
    ),
    "development_patient_count": 329,
    "test_patient_count": 83,
    "paper_series_count": 2676,
    "extended_series_count": 2680,
    "extended_frame_count": 119046,
    "multimodal_patient_count": 376,
    "split_sha256": split_sha256,
    "test_policy": (
        "Do not use TEST for preprocessing fitting, "
        "hyperparameter selection, threshold selection, "
        "early stopping, or model selection."
    ),
}

patient_split_final_df.to_csv(
    SPLIT_DIR
    / "02_patient_split_manifest.csv",
    index=False,
    encoding="utf-8-sig",
)

paper_series_split_df.to_csv(
    SPLIT_DIR
    / "03_paper_series_split_manifest.csv",
    index=False,
    encoding="utf-8-sig",
)

extended_series_split_df.to_csv(
    SPLIT_DIR
    / "04_extended_series_split_manifest.csv",
    index=False,
    encoding="utf-8-sig",
)

eligible_frame_split_df.to_csv(
    SPLIT_DIR
    / "05_frame_split_manifest.csv",
    index=False,
    encoding="utf-8-sig",
)

multimodal_patient_split_df.to_csv(
    SPLIT_DIR
    / "06_multimodal_patient_split_manifest.csv",
    index=False,
    encoding="utf-8-sig",
)

patient_distribution_df.to_csv(
    SPLIT_DIR
    / "07_patient_split_distribution.csv",
    index=False,
    encoding="utf-8-sig",
)

series_distribution_df.to_csv(
    SPLIT_DIR
    / "08_series_split_distribution.csv",
    index=False,
    encoding="utf-8-sig",
)

development_patient_fold_df.to_csv(
    SPLIT_DIR
    / "09_development_patient_fold_distribution.csv",
    index=False,
    encoding="utf-8-sig",
)

development_series_fold_df.to_csv(
    SPLIT_DIR
    / "10_development_series_fold_distribution.csv",
    index=False,
    encoding="utf-8-sig",
)

split_balance_audit_df.to_csv(
    SPLIT_DIR
    / "11_split_balance_audit.csv",
    index=False,
    encoding="utf-8-sig",
)

split_gate_df.to_csv(
    SPLIT_DIR
    / "12_split_gate.csv",
    index=False,
    encoding="utf-8-sig",
)

with open(
    SPLIT_DIR / "13_split_config.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        split_config,
        file,
        ensure_ascii=False,
        indent=2,
    )

print("Split SHA-256:", split_sha256)
print("\n✅ 환자 단위 split 최종 저장 완료")
print("✅ 앞으로 모든 모델이 이 split을 사용합니다.")

Split SHA-256: fd134b8568550dc94391ecbe6ff50074610ff9cebd0d01fe875dfcad388441d0

✅ 환자 단위 split 최종 저장 완료
✅ 앞으로 모든 모델이 이 split을 사용합니다.
